In [1]:
print("hello")

hello


In [2]:
!pip install langchain_pinecone pinecone

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain_core-0.3.86-py3-none-any.whl.metadata (3.2 kB)
Using cached langchain_core-0.3.86-py3-none-any.whl (461 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.2
    Uninstalling langchain-core-1.6.2:
      Successfully uninstalled langchain-core-1.6.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.23 requires langchain-text-splitters<1.0.0,>=0.3.8, but you have langchain-text-splitters 1.1.2 which is incompatible.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-text-splitters 1.1.2 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.3.86 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from langchain_pinecone import PineconeEmbeddings, PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
import os
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

In [14]:
PINECONE_API_KEY

'pcsk_5ijxtP_88RVh1jJrzqdauUT6WL2cDQHZWzmNRbWCyT3BDm37L9ZNfj5KuWy8SdMr4kt984'

In [15]:
pc = Pinecone(PINECONE_API_KEY)

In [17]:
index_name = 'ml-data'
if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension = 1024,
        metric = 'cosine',
        spec = ServerlessSpec(cloud = 'aws', region = 'us-east-1')
    )

In [19]:
index = pc.Index(index_name)
embedding_model = PineconeEmbeddings(model = 'multilingual-e5-large')

In [20]:
vector_store = PineconeVectorStore(index, embedding_model)

In [21]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
    id=1,
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
    id=2,
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
    id=3,
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
    id=4,
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
    id=5,
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
    id=6,
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
    id=7,
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
    id=8,
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
    id=9,
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
    id=10,
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [22]:
vector_store.add_documents(documents = documents)

['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']

In [23]:
retriever = vector_store.as_retriever(
    search_kwargs = {"k" : 4}
)

In [24]:
query = "Why iphones are overpriced"
content = retriever.invoke(query)

In [25]:
content

[Document(id='6', metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(id='10', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='3', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='8', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [26]:
vector_store.similarity_search(query, 4)

[Document(id='6', metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(id='10', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='3', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='8', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [27]:
index = pc.Index("machinelearning")

In [28]:
vector_store = PineconeVectorStore(index, embedding_model)

In [29]:
vector_store.add_documents(documents)

['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']